# 12 Training Curves — Baseline vs SDD

**Question:** Does SDD improve FID and representation quality, and does it converge faster?

We record FID + linear probe accuracy every N epochs for:
- `mse_only` (baseline, no distillation)
- `full_sdd` (all SDD components enabled)

The loss curves are also plotted to check training stability.

In [ ]:
from src.experiments import load_cfg, deep_update, compare_training_curves
import matplotlib.pyplot as plt
import pandas as pd

base_cfg = load_cfg('configs/cifar10.yaml')
base_cfg = deep_update(base_cfg, {
    'train': {'epochs': 200},  # reduce for quick test; set to 400 for full run
    'wandb': {'enabled': True, 'tags': ['cifar10', 'training_curve']},
})

variants = {
    'mse_only': {
        'sdd': {'enabled': False, 'use_centering': False, 'use_sharpening': False,
                'use_gating': False, 'use_projection_head': False, 'lambda_distill': 0.0}
    },
    'full_sdd': {
        'sdd': {'enabled': True, 'use_centering': True, 'use_sharpening': True,
                'use_gating': True, 'use_projection_head': True}
    },
}

curves = compare_training_curves(base_cfg, variants, eval_every=20)
curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
colors = {'mse_only': 'steelblue', 'full_sdd': 'darkorange'}

for name, df in curves.items():
    df_eval = df.dropna(subset=['fid'])
    c = colors[name]
    axes[0, 0].plot(df['epoch'], df['loss_total'], label=name, color=c)
    axes[0, 1].plot(df['epoch'], df['loss_mse'],   label=name, color=c)
    axes[1, 0].plot(df_eval['epoch'], df_eval['fid'],              label=name, color=c, marker='o', ms=4)
    axes[1, 1].plot(df_eval['epoch'], df_eval['linear_probe_acc'], label=name, color=c, marker='o', ms=4)

for ax, title, ylabel in zip(
    axes.flat,
    ['Total loss', 'MSE loss', 'FID ↓', 'Linear probe acc ↑'],
    ['loss', 'loss', 'FID', 'accuracy'],
):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Baseline vs SDD — training curves (CIFAR-10)', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/training_curves.png', dpi=150)
plt.show()

# Summary table
summary = {name: {'final_fid': df.dropna(subset=['fid'])['fid'].iloc[-1],
                  'final_probe_acc': df.dropna(subset=['linear_probe_acc'])['linear_probe_acc'].iloc[-1]}
           for name, df in curves.items()}
pd.DataFrame(summary).T